In [36]:
import os,sys
from src.logger import logging
from src.exception import MyException
from src.constants import DATABASE_NAME, MONGODB_URL_KEY
from dotenv import load_dotenv
import certifi
import pymongo
load_dotenv()

ca=certifi.where()
class MongoDBClient:
    client=None

    def __init__(self,database_name:str=DATABASE_NAME)->None:
        
        try:
            if MongoDBClient.client is None:
                mongodb_url=os.getenv(MONGODB_URL_KEY)
                if mongodb_url is None:
                    raise MyException(Exception,sys)
                MongoDBClient.client=pymongo.MongoClient(mongodb_url,tlsCAFile=ca)
                self.client=MongoDBClient.client
                self.database=self.client[database_name]
                self.database_name=database_name
        except Exception as e:
            raise MyException(e,sys)


In [ ]:
client=MongoDBClient()

In [37]:
from typing import Optional
import pandas as pd
import numpy as np
class get_data_from_mongo:

    def __init__(self):
        self.client=MongoDBClient(DATABASE_NAME)

    def pull_collection_data_as_dataframe(self,collection_name:str,database_name:Optional[str]=None)->pd.DataFrame:
        try:
            if database_name is None:
                collection=self.client.database[collection_name]
            collection=self.client[database_name][collection_name]

            df=pd.DataFrame(list(collection.find()))

            if '_id' in df.columns.to_list():
                df.drop('_id',axis=1)
            df.replace({'na',np.nan},inplace=True)
            return df
        
        except Exception as e:
            raise MyException(e,sys)
        
        
            



In [38]:
from src.constants import COLLECTION_NAME
obj=get_data_from_mongo()
x=obj.pull_collection_data_as_dataframe(collection_name=COLLECTION_NAME)
x

[ 2025-12-29 13:26:16,464 ] root - ERROR - Error occurred in python script: [C:\Users\krush\AppData\Local\Temp\ipykernel_17728\2409392900.py] at line number [21]: Port contains non-digit characters. Hint: username and password must be escaped according to RFC 3986, use urllib.parse.quote_plus


MyException: Error occurred in python script: [C:\Users\krush\AppData\Local\Temp\ipykernel_17728\2409392900.py] at line number [21]: Port contains non-digit characters. Hint: username and password must be escaped according to RFC 3986, use urllib.parse.quote_plus

In [ ]:
# you're going to run everything from one .py file which is pipeline.py

# so for this you have to write down a code for the same right 
import os
from dataclasses import dataclass
from datetime import datetime
TIMESTAMPS:str=datetime.now().strftime('%d-%m-%Y_%H-%M-%S')
from src.constants import *
@dataclass
class TrainingPipelineConfig:
    pipeline_name:str
    artifacts_dir:str=os.path.join(ARTIFACT_DIR,TIMESTAMPS)
    timestamp:str=TIMESTAMPS

'''what are the things we do in data ingestion -> I'll create a data_ingestion_dir in artifacts inside that I'll have 2 dirs 1 will have whole
pulled data from MONGODB another will have train and test split data'''

train_pipline_config:TrainingPipelineConfig=TrainingPipelineConfig()

@dataclass
class DataIngestionConfig:
    data_ingestion_dir:str=os.path.join(train_pipline_config.artifacts_dir,DATA_INGESTION_DIR_NAME)
    feature_store_file_path=os.path.join(data_ingestion_dir,DATA_INGESTION_FEATURE_STORE_DIR)
    train_file_path=os.path.join(data_ingestion_dir,DATA_INGESTION_INGESTED_DIR,TRAIN_FILE_NAME)
    test_file_path=os.path.join(data_ingestion_dir,DATA_INGESTION_INGESTED_DIR,TEST_FILE_NAME)
    train_test_split_ratio:float=DATA_INGESTION_TRAIN_TEST_SPLIT_RATIO
    collection_name:str=COLLECTION_NAME


In [ ]:
from src.data_access import proj1_data
from sklearn.model_selection import train_test_split
from src.entity.artifact_entity import DataIngestionArtifact
class DataIngestion:
    def __init__(self):
        self.data_ingestion_config=DataIngestionConfig()

    def export_data_into_feature_store(self)->pd.DataFrame:
        my_data=proj1_data()
        dataframe:pd.DataFrame=my_data.export_collection_as_dataframe(self.data_ingestion_config.collection_name)
        feature_dir_path=self.data_ingestion_config.feature_store_file_path
        dir_path=os.path.dirname(feature_dir_path)
        os.makedirs(dir_path,exist_ok=True)
        dataframe.to_csv(feature_dir_path,index=False,header=True)
        return dataframe
    
    def train_test_split(self,dataframe:pd.DataFrame):
        
        train_set,test_set=train_test_split(dataframe,test_size=self.data_ingestion_config.train_test_split_ratio)

        dir_path=os.path.dirname(self.data_ingestion_config.train_file_path)
        os.makedirs(dir_path,exist_ok=True)
        train_set.to_csv(self.data_ingestion_config.train_file_path,index=False,header=True)
        test_set.to_csv(self.data_ingestion_config.test_file_path,index=False,header=True)
    
    def initiate_data_ingestion(self)->DataIngestionArtifact:
        try:
            dataframe=self.export_data_into_feature_store()
            self.train_test_split(dataframe=dataframe)

            data_ingestion_artifact=DataIngestionArtifact(trained_file_path=self.data_ingestion_config.train_file_path,
                                                          test_file_path=self.data_ingestion_config.test_file_path)
            
            return data_ingestion_artifact






NameError: name 'pd' is not defined